# depends on DestinationStates

In [1]:
import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationTokenValues,
    TokenValues,
    Autopools,
    DestinationStates,
    DestinationTokens,
    Destinations,
    AutopoolDestinationStates,
    Tokens,
)


from mainnet_launch.abis import STATS_CALCULATOR_REGISTRY_ABI
from mainnet_launch.data_fetching.get_events import fetch_events


from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_orm,
    get_full_table_as_df,
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
    natural_left_right_using_where,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    identity_with_bool_success,
    get_state_by_one_block,
)
from mainnet_launch.constants import (
    ALL_CHAINS,
    ROOT_PRICE_ORACLE,
    ChainData,
    STATS_CALCULATOR_REGISTRY,
    WETH,
    ETH_CHAIN,
)

from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_autopool_to_active_destinations_over_this_period_of_missing_blocks,
)


import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationStates,
    DestinationTokenValues,
    AutopoolDestinationStates,
    Autopools,
    DestinationTokens,
    Destinations,
    Tokens,
)
from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_df,
    get_full_table_as_orm,
    insert_avoid_conflicts,
    get_highest_value_in_field_where,
    get_subset_not_already_in_column,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    get_state_by_one_block,
)
from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_active_destinations_by_autopool_by_block,
    fetch_pools_and_destinations_df,
)
from mainnet_launch.constants import (
    AutopoolConstants,
    ALL_AUTOPOOLS,
    AUTO_LRT,
    POINTS_HOOK,
    ChainData,
)


chain = ETH_CHAIN

possible_blocks = build_blocks_to_use(chain)

missing_blocks = get_subset_not_already_in_column(
    AutopoolDestinationStates,
    AutopoolDestinationStates.block,
    possible_blocks,
    where_clause=AutopoolDestinationStates.chain_id == chain.chain_id,
)

autopool_df = get_full_table_as_df(Autopools, where_clause=Autopools.chain_id == chain.chain_id)
full_destination_df = natural_left_right_using_where(
    DestinationTokens,
    Destinations,
    using=[DestinationTokens.destination_vault_address, DestinationTokens.chain_id],
    where_clause=DestinationTokens.chain_id == chain.chain_id,
)

token_value_df = natural_left_right_using_where(
    DestinationTokenValues,
    TokenValues,
    using=[DestinationTokenValues.block, DestinationTokens.chain_id, DestinationTokens.token_address],
    where_clause=DestinationTokenValues.chain_id == chain.chain_id,
)
token_df = get_full_table_as_df(Tokens, where_clause=Tokens.chain_id == chain.chain_id)
token_value_df = pd.merge(
    token_value_df, token_df[["symbol", "decimals", "token_address"]], how="left", on="token_address"
)
token_value_df

2025-04-26 21:14:48,865 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-04-26 21:14:48,866 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-26 21:14:49,060 INFO sqlalchemy.engine.Engine select current_schema()
2025-04-26 21:14:49,061 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-26 21:14:49,244 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-04-26 21:14:49,244 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-26 21:14:49,431 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-26 21:14:49,451 INFO sqlalchemy.engine.Engine SELECT max(blocks.block) AS block 
FROM blocks 
WHERE blocks.chain_id = %(chain_id_1)s AND blocks.block >= %(block_1)s AND blocks.block <= %(block_2)s GROUP BY date_trunc(%(date_trunc_1)s, blocks.datetime) ORDER BY date_trunc(%(date_trunc_2)s, blocks.datetime)
2025-04-26 21:14:49,451 INFO sqlalchemy.engine.Engine [generated in 0.00065s] {'chain_id_1': 1, 'block_1': 20752910, 'block_2': 100000000, 'date_trunc_1': 'day', 'dat

,block,chain_id,token_address,destination_vault_address,spot_price,quantity,safe_spot_spread,spot_backing_discount,denomiated_in,backing,safe_price,safe_backing_spread,symbol,decimals
0,20759464,1,0xA1290d69c65A6Fe4DF752f95823fae25cB99e5A7,0x2F7e096a400ded5D02762120b39A3aA4ABA072a4,1.023189,2.596148e+15,-0.000144,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.023337,NaN,rsETH,18
1,20759464,1,0xA1290d69c65A6Fe4DF752f95823fae25cB99e5A7,0x4E12227b350E8f8fEEc41A58D36cE2fB2e2d4575,1.023054,2.596148e+15,-0.000277,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.023337,NaN,rsETH,18
2,20759464,1,0xA1290d69c65A6Fe4DF752f95823fae25cB99e5A7,0x60339056EC88996e41757E05a798310E46972cca,1.023189,NaN,-0.000144,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.023337,NaN,rsETH,18
3,20759464,1,0xA1290d69c65A6Fe4DF752f95823fae25cB99e5A7,0xf9779aEF9f77e78C857CB4A068c65CcBee25BAAc,1.023054,NaN,-0.000277,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.023337,NaN,rsETH,18
4,20759464,1,0xD4fa2D31b7968E448877f69A96DE69f5de8cD23E,0xC064A47894349f3cc75D5edb69A2E91c0A049d4A,NaN,NaN,NaN,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,NaN,NaN,waEthUSDC,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35835,22356658,1,0xae78736Cd615f374D3085123A210448E74Fc6393,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,1.129274,7.413638e+03,0.000080,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.129184,NaN,rETH,18
35836,22356658,1,0xFAe103DC9cf190eD75350761e95403b7b8aFa6c0,0x90300b02b162F902B9629963830BcCCdeEd71113,1.038054,7.283623e+01,0.001766,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.036224,NaN,rswETH,18
35837,22356658,1,0xFAe103DC9cf190eD75350761e95403b7b8aFa6c0,0xC9b5D82652a1C8214b0971A004983d0EEeDD751C,1.038054,7.283623e+01,0.001766,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.036224,NaN,rswETH,18
35838,22356658,1,0xBe9895146f7AF43049ca1c1AE358B0541Ea49704,0x92294A62D6D9F0FbE30Ba3B543edb1806561baD7,1.095270,2.391526e-02,-0.000201,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,1.095490,NaN,cbETH,18


In [8]:
def build_autopool_balance_of_calls_by_destination(
    autopool_vault_address: str, destination_vault_addresses: list[str]
) -> list[Call]:
    return [
        Call(
            destination_vault_address,
            ["balanceOf(address)(uint256)", autopool_vault_address],
            [((autopool_vault_address, destination_vault_address), safe_normalize_with_bool_success)],
        )
        for destination_vault_address in destination_vault_addresses
    ]


# DestinationVaultAddress.underlyingTotalSupply()
#  -> I'm 95% sure this is the total supply of the lp tokens, not the quantity of lp tokens staked for rewards
def build_destinations_underlyingTotalSupply_calls(destination_vault_addresses: list[str]) -> list[Call]:
    return [
        Call(
            destination_vault_address,
            ["underlyingTotalSupply()(uint256)"],
            [(destination_vault_address, safe_normalize_with_bool_success)],
        )
        for destination_vault_address in destination_vault_addresses
    ]


def fetch_destination_total_supply_df(
    autopool_to_all_ever_active_destinations: dict, missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    all_active_destinations = set()

    for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
        this_autopool_active_destinations = [
            dest.destination_vault_address for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]
        ]
        all_active_destinations.update(this_autopool_active_destinations)

    calls = build_destinations_underlyingTotalSupply_calls(list(all_active_destinations))
    destination_total_supply_df = get_raw_state_by_blocks(calls, missing_blocks, chain, include_block_number=True)
    return destination_total_supply_df


def fetch_autopool_balance_of_by_destination(
    autopool_to_all_ever_active_destinations: dict[str, list[Destinations]], missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    autopool_balance_of_calls = []

    for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
        this_autopool_active_destinations = [
            dest.destination_vault_address for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]
        ]

        autopool_balance_of_calls.extend(
            build_autopool_balance_of_calls_by_destination(autopool_vault_address, this_autopool_active_destinations)
        )

    autopool_destination_balance_of_df = get_raw_state_by_blocks(
        autopool_balance_of_calls, missing_blocks, chain, include_block_number=True
    )
    return autopool_destination_balance_of_df


def _clean_summary_stats_info(success, summary_stats):
    if success is True:
        summary = {
            "destination": summary_stats[0],  # address
            "baseApr": summary_stats[1] / 1e18,
            "feeApr": summary_stats[2] / 1e18,
            "incentiveApr": summary_stats[3] / 1e18,
            "safeTotalSupply": summary_stats[4] / 1e18,
            "priceReturn": summary_stats[5] / 1e18,
            # "maxDiscount": summary_stats[6] / 1e18,
            # "maxPremium": summary_stats[7] / 1e18,
            # "ownedShares": summary_stats[8] / 1e18,
            "compositeReturn": summary_stats[9] / 1e18,  # in or out, ( not certain here)
            "pricePerShare": summary_stats[10] / 1e18,
            "pointsApr": None,  # set later
        }
        return summary
    else:
        return None


def _build_summary_stats_call(
    autopool: Autopools,
    dest: Destinations,
    direction: str = "out",
    amount: int = 0,
) -> Call:
    # /// @notice Gets the safe price of the underlying LP token
    # /// @dev Price validated to be inside our tolerance against spot price. Will revert if outside.
    # /// @return price Value of 1 unit of the underlying LP token in terms of the base asset
    # function getValidatedSafePrice() external returns (uint256 price);
    # getDestinationSummaryStats uses getValidatedSafePrice. So when prices are outside tolerance this function reverts

    # consider finding a version of this function that won't revert, (follow up, I am pretty sure that does not exist)
    if direction == "in":
        direction_enum = 0
    elif direction == "out":
        direction_enum = 1
    return_types = "(address,uint256,uint256,uint256,uint256,int256,int256,int256,uint256,int256,uint256)"

    # cleaning_function = build_summary_stats_cleaning_function(autopool)
    return Call(
        autopool.strategy_address,
        [
            f"getDestinationSummaryStats(address,uint8,uint256)({return_types})",
            dest.destination_vault_address,
            direction_enum,
            amount,
        ],
        [((autopool.autopool_vault_address, dest.destination_vault_address, direction), _clean_summary_stats_info)],
    )


def fetch_destination_summary_stats_df(
    autopool_to_all_ever_active_destinations: dict, missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    autopools_orm: list[Autopools] = get_full_table_as_orm(Autopools, where_clause=Autopools.chain_id == chain.chain_id)
    full_autopool_summary_stats_df = None

    for autopool in autopools_orm:
        all_summary_stats_calls = []
        this_autopool_destinations = autopool_to_all_ever_active_destinations[autopool.autopool_vault_address]
        for dest in this_autopool_destinations:
            all_summary_stats_calls.append(_build_summary_stats_call(autopool, dest, "out"))
            all_summary_stats_calls.append(_build_summary_stats_call(autopool, dest, "in"))

        autopool_summary_stats_df = get_raw_state_by_blocks(
            all_summary_stats_calls, missing_blocks, chain, include_block_number=True
        )

        if full_autopool_summary_stats_df is None:
            full_autopool_summary_stats_df = autopool_summary_stats_df.copy()
        else:
            full_autopool_summary_stats_df = pd.merge(
                full_autopool_summary_stats_df, autopool_summary_stats_df, on="block"
            )

    return full_autopool_summary_stats_df


def _fetch_autopool_and_destination_states(
    chain: ChainData,
    missing_blocks: list[int],
):
    autopool_df = get_full_table_as_df(Autopools, where_clause=Autopools.chain_id == chain.chain_id)

    full_destination_df = natural_left_right_using_where(
        DestinationTokens,
        Destinations,
        using=[DestinationTokens.destination_vault_address, DestinationTokens.chain_id],
        where_clause=DestinationTokens.chain_id == chain.chain_id,
    )

    token_value_df = natural_left_right_using_where(
        DestinationTokenValues,
        TokenValues,
        using=[DestinationTokenValues.block, DestinationTokens.chain_id, DestinationTokens.token_address],
        where_clause=DestinationTokenValues.chain_id == chain.chain_id,
    )
    # not sure here on the way to specify only a subset of oclumsn? maybe with type hints? add later, eg columsn = none and
    token_df = get_full_table_as_df(Tokens, where_clause=Tokens.chain_id == chain.chain_id)[
        ["symbol", "decimals", "token_address"]
    ]
    token_value_df = pd.merge(token_value_df, token_df, how="left", on="token_address")

    autopool_to_all_ever_active_destinations = fetch_autopool_to_active_destinations_over_this_period_of_missing_blocks(
        chain, missing_blocks
    )

    destination_underlying_total_supply_df = fetch_destination_total_supply_df(
        autopool_to_all_ever_active_destinations, missing_blocks, chain
    )
    autopool_destination_balance_of_df = fetch_autopool_balance_of_by_destination(
        autopool_to_all_ever_active_destinations, missing_blocks, chain
    )  # only needed for autopool destination state, the rest can be extracted from the other tables

    autopool_summary_stats_df = fetch_destination_summary_stats_df(
        autopool_to_all_ever_active_destinations, missing_blocks, chain
    )
    return (
        autopool_summary_stats_df,
        destination_underlying_total_supply_df,
        token_value_df,
        autopool_to_all_ever_active_destinations,
    )


(
    autopool_summary_stats_df,
    destination_underlying_total_supply_df,
    token_value_df,
    autopool_to_all_ever_active_destinations,
) = _fetch_autopool_and_destination_states(chain, missing_blocks)

2025-04-26 21:24:52,641 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-26 21:24:52,642 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2025-04-26 21:24:52,642 INFO sqlalchemy.engine.Engine [cached since 602.6s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x14596d120>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2025-04-26 21:24:52,643 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM autopools
            WHERE autopools.chain_id = 1
        
2025-04-26 21:24:52,643 INFO

[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]

In [ ]:
all_new_destination_states = []
# autopool_summary_stats_df, destination_underlying_total_supply_df, token_value_df, autopool_to_all_ever_active_destinations
raw_destination_states_df = pd.merge(autopool_summary_stats_df, destination_underlying_total_supply_df, on="block")

for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
    for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]:

        underlying_total_supply_series = raw_destination_states_df[dest.destination_vault_address]
        in_summary_stats_series = raw_destination_states_df[
            (autopool_vault_address, dest.destination_vault_address, "in")
        ]
        out_summary_stats_series = raw_destination_states_df[
            (autopool_vault_address, dest.destination_vault_address, "out")
        ]

        sub_
        break

In [ ]:
# maybe we get the apply map version?

In [ ]:
# class DestinationStates(Base):
#     __tablename__ = "destination_states"

#     destination_vault_address: Mapped[str] = mapped_column(primary_key=True)
#     block: Mapped[int] = mapped_column(primary_key=True)
#     chain_id: Mapped[int] = mapped_column(primary_key=True)
#     # information about the destination itself at this moment in time

#     incentive_apr: Mapped[float] = mapped_column(nullable=False)
#     fee_and_base_apr: Mapped[float] = mapped_column(nullable=False)
#     points_apr: Mapped[float] = mapped_column(nullable=True)

#     total_apr_in: Mapped[float] = mapped_column(
#         nullable=True
#     )  # get destination summaryStats (in, and out) are seperate calls
#     total_apr_out: Mapped[float] = mapped_column(nullable=True)

#     underlying_token_total_staked: Mapped[float] = mapped_column(nullable=True)
#     underlying_token_total_supply: Mapped[float] = mapped_column(nullable=False)
#     safe_total_supply: Mapped[float] = mapped_column(nullable=True)  # only for pre autoUSD destinations

#     # this is as lp tokens # via
#     underlying_safe_price: Mapped[float] = mapped_column(nullable=False)
#     underlying_spot_price: Mapped[float] = mapped_column(nullable=False)
#     underlying_backing: Mapped[float] = mapped_column(nullable=False)
#     denominated_in: Mapped[str] = mapped_column(nullable=False) # should live in the destination

#     safe_backing_discount: Mapped[float] = mapped_column(nullable=True)
#     safe_spot_spread: Mapped[float] = mapped_column(nullable=True)
#     spot_backing_discount: Mapped[float] = mapped_column(nullable=True)

#     __table_args__ = (
#         ForeignKeyConstraint(["block", "chain_id"], ["blocks.block", "blocks.chain_id"]),
#         ForeignKeyConstraint(
#             ["destination_vault_address", "chain_id"],
#             ["destinations.destination_vault_address", "destinations.chain_id"],
#         ),
#     )

[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]


,0x2B08137BeABd2454AD3631DEB754F97C5c93eB78,0xe4433D00Cf48BFE0C672d9949F2cd2c008bffC04,0x5A4B544B9734930DDC587c9a2f093dC5058A4f4D,0xdfE3fA7027E84f59b266459C567278C79fe86f0C,0x148Ca723BefeA7b021C399413b8b7426A4701500,0xE382BBd32C4E202185762eA433278f4ED9E6151E,0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2,0xd96E943098B2AE81155e98D7DC8BeaB34C539f01,0x356c79Ab2B2cEFAB685004CE827146058A6c3e77,0x6a8C6ff78082a2ae494EB9291DdC7254117298Ff,...,"(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x2f2CC1bf461413014741dD68481dB4a3686DAC3D)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x40219bBda953ca811d2D0168Dc806a96b84791d9)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0xC9b5D82652a1C8214b0971A004983d0EEeDD751C)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0xf9779aEF9f77e78C857CB4A068c65CcBee25BAAc)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x60339056EC88996e41757E05a798310E46972cca)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x5c6aeb9ef0d5BbA4E6691f381003503FD0D45126)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, 0xc4Eb861e7b66f593482a3D7E8adc314f6eEDA30B)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, 0xe4433D00Cf48BFE0C672d9949F2cd2c008bffC04)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, 0xbA1462f43c6f60ebD1C62735c94E428aD073E01A)",block
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-09-15 23:59:59+00:00,NaN,NaN,NaN,NaN,2550.248990,3004.943853,352.135269,4418.771443,NaN,84.650070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20759464
2024-09-16 23:59:59+00:00,NaN,NaN,NaN,NaN,2550.306071,3024.939003,352.151418,4418.771443,NaN,84.650070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20766617
2024-09-17 23:59:59+00:00,NaN,NaN,NaN,NaN,2550.375275,3124.928602,484.989135,4414.993123,NaN,84.650070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20773761
2024-09-18 23:59:59+00:00,NaN,NaN,NaN,NaN,2550.571438,3389.359227,485.059853,4561.283097,NaN,84.650070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20780916
2024-09-19 23:59:59+00:00,NaN,NaN,NaN,NaN,2550.672146,3724.974425,485.091132,4598.273271,NaN,84.650070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20788084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-22 23:59:59+00:00,227.502467,8023.442403,12327.821057,684.921261,1628.949736,6199.358769,0.784321,8023.442403,5942.370219,227.502467,...,0.0,0.0,0.0,0.0,0.0,470.519208,317.661117,0.0,540.652839,22327989
2025-04-23 23:59:59+00:00,227.502467,8078.410498,12328.542843,688.107559,1629.680081,6029.808420,0.784354,8078.410498,5942.370219,227.502467,...,0.0,0.0,0.0,0.0,0.0,470.519208,317.661117,0.0,540.652839,22335153
2025-04-24 23:59:59+00:00,264.159941,8052.117527,11642.586887,734.304846,1629.830873,6079.772900,0.784384,8052.117527,5942.370219,264.159941,...,0.0,0.0,0.0,0.0,0.0,470.519208,317.661117,0.0,540.652839,22342307


In [ ]:
AutopoolDestinationStates_new_partial_rows = []

def _to_apply_over_each_row(row:dict):
    
    for autopool_dest_tuple, balance_of in row.items():
        if autopool_dest_tuple != 'block':
            autopool_vault_address, destination_vault_address = col # unpack the tuple

            AutopoolDestinationStates_new_partial_rows.append(
                        {
            'autopool_vault_address':autopool_vault_address,
            'destination_vault_address':destination_vault_address, 
            'block': row['block'],
            'amount': balance_of,
            # this will raise an error if trying to insert into AutopoolDestinationStates because nullable = false 
            'total_safe_value': None,
            'total_spot_value': None,
            'total_backing_value': None,
            'percent_ownership': -100.0, # depends on destination_states.underlying_token_total_supply
        }

            )

# t



for col in autopool_destination_balance_of_df.columns:
    this_autopool_destination_balance_of = []
    autopool_balance_columns = autopool_destination_balance_of_df[col]
    autopool_vault_address, destination_vault_address = col # unpack
    this_autopool_destination_balance_of.append(
        {
            'autopool_vault_address':autopool_vault_address,
            'destination_vault_address':

        }

    )



0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xf3ae3c74EaD129e770A864CeE291A805b170bBe0
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x865e59D439BF7310c9BC6117E6020B8C87De4065
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xe3AE2Ab9AE8ADe1B4940dd893C9339401bEe61A1
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xfB6f99FdF12E37Bfe3c4Cf81067faB10c465fb24
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x8cA2201BC34780f14Bca452913ecAc8e9928d4cA
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x896eCc16Ab4AFfF6cE0765A5B924BaECd7Fa455a
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xC001f23397dB71B17602Ce7D90a983Edc38DB0d1
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x6a8C6ff78082a2ae494EB9291DdC7254117298Ff
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xd96E943098B2AE81155e98D7DC8BeaB34C539f01
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xE382BBd32

In [ ]:
autopool_to_all_ever_active_destinations.keys()

dict_keys(['0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', '0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5', '0xE800e3760FC20aA98c5df6A9816147f190455AF3', '0x35911af1B570E26f668905595dEd133D01CD3E5a'])

In [ ]:
autopool_to_all_ever_active_destinations.keys

In [ ]:
# def build_pool_token_spot_price_calls(
#     chain: ChainData, pool_addresses: list[str], token_addresses: list[str]
# ) -> list[Call]:

#     return [
#         Call(
#             ROOT_PRICE_ORACLE(chain),
#             ["getSpotPriceInEth(address,address)(uint256)", token_address, pool_address],
#             [(f"{pool_address}_{token_address}", safe_normalize_with_bool_success)],
#         )
#         for (pool_address, token_address) in zip(pool_addresses, token_addresses)
#     ]


# spot_price_calls = build_pool_token_spot_price_calls(
#     chain, full_destination_df["pool"], full_destination_df["token_address"]
# )
# destination_token_spot_price_df = get_raw_state_by_blocks(
#     spot_price_calls, possible_blocks, chain, include_block_number=True
# )
# destination_token_spot_price_df


# def build_underlying_reserves_calls(destinations: list[str]) -> list[Call]:
#     return [
#         Call(
#             dest,
#             "underlyingReserves()(address[],uint256[])",
#             [
#                 (f"{dest}_underlyingReserves_tokens", identity_with_bool_success),
#                 (f"{dest}_underlyingReserves_amounts", identity_with_bool_success),
#             ],
#         )
#         for dest in destinations
#     ]


# underlying_reserves_calls = build_underlying_reserves_calls(full_destination_df["destination_vault_address"])
# underlying_reserves_df = get_raw_state_by_blocks(
#     underlying_reserves_calls, possible_blocks, chain, include_block_number=True
# )
# underlying_reserves_df